# RAG + Qwen: Ad Marketing Analysis

This notebook runs **Retrieval-Augmented Generation** using **Qwen2** and your ad datasets. Designed for **Google Colab** with GPU.

## 1. Install Dependencies

In [ ]:
!pip install -q pandas chromadb sentence-transformers transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60

## 2. GPU Check & Data Paths

In [ ]:
import os
import pandas as pd

# Check GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Upload CSVs via Colab file picker (left sidebar) - they go to /content
DATA_DIR = "/content"

# File paths (adjust if your filenames differ)
ADS_CSV = os.path.join(DATA_DIR, "collected_ads_enriched.csv")
ADS_FALLBACK = os.path.join(DATA_DIR, "collected_ads.csv")
NLP_CSV = os.path.join(DATA_DIR, "nlp", "image_text_analysis.csv")
STRATEGY_CSV = os.path.join(DATA_DIR, "ads_with_strategies.csv")

# If nlp is in same dir as ads
if not os.path.exists(NLP_CSV):
    NLP_CSV = os.path.join(DATA_DIR, "image_text_analysis.csv")

if not os.path.exists(ADS_CSV):
    ADS_CSV = ADS_FALLBACK

print(f"Ads file: {ADS_CSV} (exists: {os.path.exists(ADS_CSV)})")
print(f"NLP file: {NLP_CSV} (exists: {os.path.exists(NLP_CSV)})")

GPU available: True
GPU: Tesla T4
Ads file: /content/collected_ads_enriched.csv (exists: True)
NLP file: /content/image_text_analysis.csv (exists: True)


## 3. Load & Merge Data

In [ ]:
ads = pd.read_csv(ADS_CSV)
print(f"Loaded {len(ads)} ads from {ADS_CSV}")

# Merge NLP analysis (OCR + sentiment + keywords)
if os.path.exists(NLP_CSV):
    nlp = pd.read_csv(NLP_CSV)
    merge_cols = ["ad_id", "sentiment_polarity", "sentiment_subjectivity", "top_words", "top_keywords"]
    if "ocr_text" not in ads.columns and "ocr_text" in nlp.columns:
        merge_cols.append("ocr_text")
    merge_cols = [c for c in merge_cols if c in nlp.columns]
    ads = ads.merge(nlp[merge_cols], on="ad_id", how="left")
    if "top_keywords" in ads.columns:
        print(f"Merged NLP data: {ads['top_keywords'].notna().sum()} ads have keywords")
    else:
        print("Merged NLP data (no 'top_keywords' column found after merge).")
else:
    print("NLP CSV not found; skipping NLP merge.")

# Merge strategy clustering (cluster + high-level strategy label)
if os.path.exists(STRATEGY_CSV):
    strat = pd.read_csv(STRATEGY_CSV)
    extra_cols = []
    if "cluster" in strat.columns and "cluster" not in ads.columns:
        extra_cols.append("cluster")
    if "strategy" in strat.columns and "strategy" not in ads.columns:
        extra_cols.append("strategy")
    if extra_cols:
        merge_cols = ["ad_id"] + extra_cols
        ads = ads.merge(strat[merge_cols], on="ad_id", how="left")
        if "strategy" in ads.columns:
            print(f"Merged strategy data: {ads['strategy'].notna().sum()} ads have strategy labels")
        else:
            print("Merged strategy data (no 'strategy' column found after merge).")
    else:
        print("Strategy CSV found but ads already contain 'cluster'/'strategy'; no extra merge needed.")
else:
    print("Strategy CSV not found; skipping strategy merge.")

ads.head(2)

Loaded 5455 ads from /content/collected_ads_enriched.csv
Merged NLP data: 2410 ads have keywords
Merged strategy data: 2047 ads have strategy labels


,ad_id,json_key,image_path,competitor,all_categories,all_categories_full,all_sentiments,all_sentiments_full,objects_symbols,image_width,...,dominant_color_4,dominant_color_5,color_palette_json,extraction_status,sentiment_polarity,sentiment_subjectivity,top_words,top_keywords,cluster,strategy
0,170489,10/170489.png,./10/170489.png,chocolate,chocolate,"Chocolate, cookies, candy, ice cream",creative|amused|conscious|eager,"Creative(inventive,productive)|Amused(humored,...",irony/comedy|entertainment,1024,...,#F5CA45,#9F3B20,"[""#2B2C39"", ""#E98533"", ""#D8D0DC"", ""#F5CA45"", ""...",success,NaN,NaN,NaN,NaN,NaN,NaN
1,174042,10/174042.png,./10/174042.png,restaurant,restaurant,"Restaurants, cafe, fast food",creative|amused|alert|conscious,"Creative(inventive,productive)|Amused(humored,...",excuses|excuses|Home/Awake|Excuses|hazard,600,...,#BC2428,#947056,"[""#F6BA34"", ""#2D2726"", ""#DCD8D8"", ""#BC2428"", ""...",success,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Create Documents for RAG

In [ ]:
def row_to_document(row):
    """Convert an ad row into a searchable text document."""
    parts = [f"Ad ID: {row.get('ad_id', '')}"]

    ocr = row.get("ocr_text", "")
    if pd.notna(ocr) and str(ocr).strip():
        parts.append(f"OCR text: {str(ocr)[:500]}")

    cat = row.get("all_categories_full", "")
    if pd.notna(cat):
        parts.append(f"Categories: {cat}")

    sent = row.get("all_sentiments_full", "")
    if pd.notna(sent):
        parts.append(f"Sentiments: {sent}")

    comp = row.get("competitor", "")
    if pd.notna(comp):
        parts.append(f"Competitor: {comp}")

    kw = row.get("top_keywords", "")
    if pd.notna(kw):
        parts.append(f"Keywords: {kw}")

    layout = row.get("layout_type", "")
    if pd.notna(layout):
        parts.append(f"Layout: {layout}")

    colors = [c for c in [row.get(f"dominant_color_{i}", "") for i in range(1, 6)] if pd.notna(c) and str(c).strip()]
    if colors:
        parts.append("Colors: " + ", ".join(colors))

    strat = row.get("strategy", "")
    if pd.notna(strat) and str(strat).strip():
        parts.append(f"Strategy label: {str(strat)}")

    cluster = row.get("cluster", "")
    if pd.notna(cluster) and str(cluster) != "":
        parts.append(f"Strategy cluster: {cluster}")

    return " | ".join(parts)

documents = [row_to_document(ads.iloc[i]) for i in range(len(ads))]
metadatas = [{"ad_id": str(ads.iloc[i]["ad_id"])} for i in range(len(ads))]

print(f"Created {len(documents)} documents")
print("\nExample document:")
print(documents[0][:400] + "...")

Created 5455 documents

Example document:
Ad ID: 170489 | OCR text: N | Categories: Chocolate, cookies, candy, ice cream | Sentiments: Creative(inventive,productive)|Amused(humored,laughing)|Conscious(aware,thoughtful,prepared)|Eager(hungry,thirsty,passionate) | Competitor: chocolate | Layout: image_heavy | Colors: #2B2C39, #E98533, #D8D0DC, #F5CA45, #9F3B20...


## 5. Build Vector Store (Embeddings)

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

# Use a fast, multilingual model (works on CPU too, GPU speeds it up)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding documents...")
embeddings = embed_model.encode(documents, show_progress_bar=True)

# Create ChromaDB collection
client = chromadb.PersistentClient(path="./vector_db")
collection = client.get_or_create_collection(
    name="ads",
    metadata={"hnsw:space": "cosine"}
)

#auto update embeddings
existing_ids = set(collection.get()["ids"])

new_docs = []
new_ids = []

for i,row in ads.iterrows():
    ad_id=str(row["ad_id"])

    if ad_id not in existing_ids:

        doc=row_to_document(row)

        new_docs.append(doc)
        new_ids.append(ad_id)

embeddings = embed_model.encode(new_docs)

collection.add(
 ids=new_ids,
 embeddings=embeddings.tolist(),
 documents=new_docs
)

print(f"Indexed {collection.count()} documents in ChromaDB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding documents...


Batches:   0%|          | 0/171 [00:00<?, ?it/s]

Indexed 5455 documents in ChromaDB


## 6. Load Qwen LLM

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Qwen2-7B loaded with 4-bit quantization")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2-7B loaded with 4-bit quantization


## 7. RAG Query Function

In [ ]:
def rag_query(question: str, top_k: int = 5, max_new_tokens: int = 256):
    """Retrieve relevant ads and generate an answer with Qwen."""
    # Retrieve
    q_embedding = embed_model.encode([question])[0].tolist()
    results = collection.query(query_embeddings=[q_embedding], n_results=top_k)

    context = "\n\n---\n\n".join(results["documents"][0])

    prompt=f"""
You are an expert in advertising strategy analysis.

Use the ad data below to answer the question.

Context:
{context}

Question:
{question}

Provide:
1. Key insights
2. Industry patterns
3. Strategic interpretation
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

## 8. Run Queries

In [ ]:
# Ask questions that use ad data + CV ad analysis + NLP + strategy clustering

your_question = "What themes appear in beauty and fragrance ads?"
print(rag_query(your_question))




Themes that appear in beauty and fragrance ads include:

- **Creative**: These ads often showcase innovative products or creative use of products to achieve unique looks or results.
- **Amused**: Humor is used to engage viewers and make the brand memorable.
- **Emotional**: Ads may evoke feelings such as vulnerability, nostalgia, or a sense of emotional connection to the product.
- **Loving**: Romantic or affectionate themes can be used to highlight the caring nature of beauty products.
- **Fashionable**: Emphasizing trends and elegance to appeal to style-conscious consumers.
- **Calm**: Creating a serene or soothing atmosphere to promote relaxation or well-being.
- **Feminine**: Highlighting qualities traditionally associated with femininity such as womanliness or girlishness.
- **Inspired**: Motivating or empowering messages to inspire confidence or ambition.
- **Cheerful**: Using happiness and positivity to uplift the viewer's mood.
- **Alarmed**: Sometimes, ads aim to raise awarene

In [ ]:
#2026 哪个品牌如何如何 数据检索 时间品牌行业 A的广告占比
#数据分析 天然主题占比。。。对比不同品牌比B高百分之12 RAG qwen调用实现 sql语句
#广告创造功能 同类高转化 指导未来 qwen高转化广告特征 生成 多模态理解 generate ad copy def 风格木质emoji 文本生成api 构造产品目标人群生成高光文案